<a href="https://colab.research.google.com/github/lucashobbs17/Geostorm/blob/main/notebooks/01_flare_cme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [203]:
   !pip install -q "sunpy[net]"

   from google.colab import drive
   drive.mount('/content/drive')

   import os
   import pandas as pd
   import numpy as np
   import matplotlib.pyplot as plt

   DATA = '/content/drive/MyDrive/geostorm/data'
   os.makedirs(DATA, exist_ok=True)
   print("Setup done")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup done


   ## 2. Get flares (GOES via HEK)

In [204]:
from sunpy.net import Fido, attrs as a

YEARS = range(2010, 2021)
cols = ["event_starttime", "event_peaktime", "event_endtime",
        "fl_goescls", "hgs_x", "hgs_y"]

parts = []
for yr in YEARS:
    path = f'{DATA}/flares_{yr}.csv'
    if os.path.exists(path):
        parts.append(pd.read_csv(path))
    else:
        res = Fido.search(
            a.Time(f"{yr}-01-01", f"{yr}-12-31"),
            a.hek.EventType("FL"),
            a.hek.FL.GOESCls > "M1.0",
            a.hek.OBS.Observatory == "GOES",
        )
        tbl = res["hek"]
        if len(tbl) == 0:
            df = pd.DataFrame(columns=cols)
        else:
            df = tbl[cols].to_pandas()
        df.to_csv(path, index=False)
        parts.append(df)
        print(f"{yr}: downloaded {len(df)}")

flares = pd.concat(parts, ignore_index=True)
print("Total flares:", len(flares))

Total flares: 716


In [205]:
print(flares.shape)
flares["fl_goescls"].value_counts()

(716, 6)


,count
fl_goescls,
M1.1,70
M1.3,62
M1.2,56
M1.4,44
M1.6,30
...,...
M8.2,1
X2.7,1
X9.3,1


In [206]:
missing = (flares.hgs_x == 0) & (flares.hgs_y == 0)
print("Missing locations:", missing.sum())

dupes = flares[flares.duplicated("event_peaktime", keep=False)]
print("Duplicate rows:", len(dupes))
dupes

Missing locations: 321
Duplicate rows: 15


,event_starttime,event_peaktime,event_endtime,fl_goescls,hgs_x,hgs_y
16,2010-11-04 23:30:00,2010-11-04 23:58:00,2010-11-05 00:12:00,M1.6,-76,-20
17,2010-11-04 23:30:00,2010-11-04 23:58:00,2010-11-05 00:12:00,M1.6,-76,-20
71,2011-09-06 22:12:00,2011-09-06 22:20:00,2011-09-06 22:24:00,X2.1,18,14
72,2011-09-06 22:12:00,2011-09-06 22:20:00,2011-09-06 22:24:00,X2.1,18,14
349,2014-01-04 10:16:00,2014-01-04 10:25:00,2014-01-04 10:41:00,M1.3,-48,-5
350,2014-01-04 10:16:00,2014-01-04 10:25:00,2014-01-04 10:41:00,M1.3,-48,-5
351,2014-01-04 18:47:00,2014-01-04 19:46:00,2014-01-04 20:23:00,M4.0,0,0
352,2014-01-04 19:05:00,2014-01-04 19:46:00,2014-01-04 20:23:00,M4.0,0,0
363,2014-01-28 12:33:00,2014-01-28 12:46:00,2014-01-28 12:50:00,M1.3,0,0
364,2014-01-28 12:38:00,2014-01-28 12:46:00,2014-01-28 12:50:00,M1.3,0,0


In [207]:
flares.loc[missing, ["hgs_x", "hgs_y"]] = np.nan
flares = flares.drop_duplicates("event_peaktime").reset_index(drop=True)
print(flares.shape)

(708, 6)


In [208]:
raw = pd.concat([pd.read_csv(f'{DATA}/flares_{y}.csv') for y in YEARS], ignore_index=True)
print("Downloaded:", len(raw))
print("After dedupe:", len(raw.drop_duplicates("event_peaktime")))
print("Final:", len(flares))

Downloaded: 716
After dedupe: 708
Final: 708


In [209]:
for c in ["event_starttime", "event_peaktime", "event_endtime"]:
    flares[c] = pd.to_datetime(flares[c])

flares = flares.sort_values("event_peaktime").reset_index(drop=True)

In [210]:
cols_ssw = ["event_peaktime", "hgs_x", "hgs_y"]

parts = []
for yr in YEARS:
    path = f'{DATA}/ssw_locations_{yr}.csv'
    if os.path.exists(path):
        parts.append(pd.read_csv(path))
    else:
        res2 = Fido.search(
            a.Time(f"{yr}-01-01", f"{yr}-12-31"),
            a.hek.EventType("FL"),
            a.hek.FRM.Name == "SSW Latest Events",
        )
        tbl = res2["hek"]
        if len(tbl) == 0:
            df = pd.DataFrame(columns=cols_ssw)
        else:
            df = tbl[cols_ssw].to_pandas()
        df.to_csv(path, index=False)
        parts.append(df)
        print(f"{yr}: {len(df)} SSW records")

ssw = pd.concat(parts, ignore_index=True)
ssw["event_peaktime"] = pd.to_datetime(ssw["event_peaktime"])
ssw = ssw.sort_values("event_peaktime").reset_index(drop=True)
print("Total SSW records:", len(ssw))

Total SSW records: 18607


In [211]:
print("SSW records:", len(ssw))
print(ssw.event_peaktime.dt.year.value_counts().sort_index())

SSW records: 18607
event_peaktime
2010    1341
2011    2546
2012    2493
2013    2398
2014    2629
2015    2291
2016    1382
2017    1257
2018     534
2019     467
2020    1269
Name: count, dtype: int64


In [212]:
for c in ["event_starttime", "event_peaktime", "event_endtime"]:
    flares[c] = pd.to_datetime(flares[c])

for c in ["hgs_x", "hgs_y"]:
    flares[c] = pd.to_numeric(flares[c], errors="coerce")

flares = flares.sort_values("event_peaktime").reset_index(drop=True)

for c in ["hgs_x", "hgs_y"]:
    ssw[c] = pd.to_numeric(ssw[c], errors="coerce")

zero_ssw = (ssw.hgs_x == 0) & (ssw.hgs_y == 0)
ssw.loc[zero_ssw, ["hgs_x", "hgs_y"]] = np.nan

merged = pd.merge_asof(
    flares,
    ssw.rename(columns={"hgs_x": "ssw_x", "hgs_y": "ssw_y"}),
    on="event_peaktime",
    direction="nearest",
    tolerance=pd.Timedelta("10min"),
)

flares["hgs_x"] = flares["hgs_x"].fillna(merged["ssw_x"])
flares["hgs_y"] = flares["hgs_y"].fillna(merged["ssw_y"])
print("Still missing:", flares["hgs_x"].isna().sum())

Still missing: 10


In [213]:

zero = (flares.hgs_x == 0) & (flares.hgs_y == 0)
print("Zero locations found:", zero.sum())
flares.loc[zero, ["hgs_x", "hgs_y"]] = np.nan

flares = flares.dropna(subset=["hgs_x", "hgs_y"]).reset_index(drop=True)
flares.to_csv(f'{DATA}/flares_clean.csv', index=False)
print(flares.shape)

Zero locations found: 0
(698, 6)


In [214]:
print("After concat:", len(flares))
print("Duplicate peak times:", flares.duplicated("event_peaktime").sum())
print("Zero locations:", ((flares.hgs_x == 0) & (flares.hgs_y == 0)).sum())

After concat: 698
Duplicate peak times: 0
Zero locations: 0


In [215]:
print("Rows:", len(flares))
print("Any missing:", flares.hgs_x.isna().sum())
print(flares.event_starttime.dt.year.value_counts().sort_index())


Rows: 698
Any missing: 0
event_starttime
2010     17
2011    100
2012    118
2013     98
2014    197
2015    113
2016     12
2017     41
2020      2
Name: count, dtype: int64


## 3. Get CMEs (CDAW LASCO catalogue)


In [216]:
import requests

path_cme = f'{DATA}/cdaw_univ_all.txt'

if not os.path.exists(path_cme):
    url = "https://cdaw.gsfc.nasa.gov/CME_list/UNIVERSAL/text_ver/univ_all.txt"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(path_cme, "w") as f:
        f.write(r.text)
    print("Downloaded")
else:
    print("Already saved")

Already saved


In [217]:
rows = []
with open(path_cme) as f:
    for line in f:
        parts = line.split()
        if len(parts) >= 5 and "/" in parts[0] and parts[0][:4].isdigit():
            rows.append(parts[:5])

cmes = pd.DataFrame(rows, columns=["date", "time", "cpa", "width", "speed"])
cmes["cme_time"] = pd.to_datetime(cmes["date"] + " " + cmes["time"])

cmes["halo"] = cmes["cpa"] == "Halo"

for c in ["cpa", "width", "speed"]:
    cmes[c] = pd.to_numeric(cmes[c], errors="coerce")

cmes = cmes.drop(columns=["date", "time"])

In [218]:
print("Before drop:", len(flares))

Before drop: 698


In [219]:
print(cmes.shape)
print("Halo CMEs:", cmes["halo"].sum())
cmes.head()

(43140, 5)
Halo CMEs: 1024


,cpa,width,speed,cme_time,halo
0,267.0,18,499.0,1996-01-11 00:14:36,False
1,265.0,16,290.0,1996-01-13 22:08:30,False
2,262.0,43,525.0,1996-01-15 07:01:10,False
3,105.0,37,267.0,1996-01-22 03:11:01,False
4,90.0,27,262.0,1996-01-26 09:16:19,False


## 4. Match flares to CMEs (labels)

In [220]:
lon = np.radians(flares["hgs_x"])
lat = np.radians(flares["hgs_y"])

x = np.cos(lat) * np.sin(lon)   # east-west on the disk (west positive)
y = np.sin(lat)                 # north-south on the disk

flares["flare_pa"] = np.degrees(np.arctan2(-x, y)) % 360
flares["dist_from_centre"] = np.sqrt(x**2 + y**2)

flares[["fl_goescls", "hgs_x", "hgs_y", "flare_pa", "dist_from_centre"]].head()

,fl_goescls,hgs_x,hgs_y,flare_pa,dist_from_centre
0,M1.7,-88.0,-28.0,118.014475,0.999525
1,M1.6,-88.0,-24.0,114.012976,0.999492
2,M1.8,-87.0,-25.0,115.030110,0.998874
3,M3.4,-81.0,-26.0,116.280724,0.990066
4,M2.9,-17.0,21.0,37.294869,0.450478


In [221]:
def angle_diff(a, b):
    d = abs(a - b) % 360
    return np.minimum(d, 360 - d)

WINDOW = pd.Timedelta("60min")
MAX_ANGLE = 45

def match_flares(flares, cmes, shift=pd.Timedelta(0)):
    labels = []
    for _, f in flares.iterrows():
        start = f.event_starttime + shift
        cand = cmes[(cmes.cme_time >= start) &
                    (cmes.cme_time <= start + WINDOW)]
        ok = cand[cand.halo | (angle_diff(cand.cpa, f.flare_pa) <= MAX_ANGLE)]
        labels.append(int(len(ok) > 0))
    return np.array(labels)

flares["cme"] = match_flares(flares, cmes)
print(flares["cme"].value_counts())


cme
0    423
1    275
Name: count, dtype: int64


In [222]:
flares.to_csv(f'{DATA}/flares_labelled.csv', index=False)

Realised benefit from 60-90 produced as much noise as real flares, so the window was shortened


In [223]:
for h in [-24, -12, 12, 24]:
    fake = match_flares(flares, cmes, pd.Timedelta(hours=h))
    print(f"Shift {h:+}h: {fake.mean():.0%} matched")

Shift -24h: 11% matched
Shift -12h: 12% matched
Shift +12h: 12% matched
Shift +24h: 9% matched


In [224]:
shifts = [pd.Timedelta(hours=h) for h in [-24, -12, 12, 24]]

for w in [45, 60, 90]:
    for ang in [30, 45]:
        WINDOW = pd.Timedelta(minutes=w)
        MAX_ANGLE = ang
        real = match_flares(flares, cmes).mean()
        chance = np.mean([match_flares(flares, cmes, s).mean() for s in shifts])
        print(f"{w:>3} min, {ang}°: real {real:.0%}, chance {chance:.0%}, "
              f"gap {real - chance:.0%}")

WINDOW = pd.Timedelta("90min")   # reset to the originals
MAX_ANGLE = 45

 45 min, 30°: real 28%, chance 7%, gap 21%
 45 min, 45°: real 31%, chance 8%, gap 22%
 60 min, 30°: real 36%, chance 9%, gap 27%
 60 min, 45°: real 39%, chance 11%, gap 28%
 90 min, 30°: real 43%, chance 14%, gap 29%
 90 min, 45°: real 47%, chance 17%, gap 30%


## 5. Build features

In [225]:
scale = {"C": 1e-6, "M": 1e-5, "X": 1e-4}
flares["peak_flux"] = (flares["fl_goescls"].str[0].map(scale)
                       * flares["fl_goescls"].str[1:].astype(float))
flares["log_flux"] = np.log10(flares["peak_flux"])

flares["duration_min"] = (flares["event_endtime"] - flares["event_starttime"]).dt.total_seconds() / 60
flares["rise_min"] = (flares["event_peaktime"] - flares["event_starttime"]).dt.total_seconds() / 60
flares["decay_min"] = flares["duration_min"] - flares["rise_min"]


In [226]:
FEATURES = ["log_flux", "dist_from_centre", "decay_min"]

print(flares[FEATURES].describe().round(2))
print()
print(flares.groupby("cme")[FEATURES].mean().round(2))

       log_flux  dist_from_centre  decay_min
count    698.00            698.00     698.00
mean      -4.61              0.72      12.91
std        0.35              0.26      15.52
min       -4.96              0.07       1.00
25%       -4.89              0.53       5.00
50%       -4.72              0.78       8.00
75%       -4.43              0.96      15.00
max       -3.03              1.00     178.00

     log_flux  dist_from_centre  decay_min
cme                                       
0       -4.69              0.69       11.3
1       -4.48              0.76       15.4


##6 Train model

In [227]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = flares[FEATURES]
y = flares["cme"]

model = make_pipeline(StandardScaler(), LogisticRegression())
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print("Baseline (always 'no CME'):", round(1 - y.mean(), 2))
print("Accuracy per fold:", scores.round(2))
print("Mean accuracy:", round(scores.mean(), 2))

Baseline (always 'no CME'): 0.61
Accuracy per fold: [0.69 0.67 0.66 0.67 0.67]
Mean accuracy: 0.67


## 7 Evaluate

In [228]:
auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
print("AUC per fold:", auc.round(2))
print("Mean AUC:", round(auc.mean(), 2))

AUC per fold: [0.76 0.71 0.69 0.7  0.69]
Mean AUC: 0.71


In [229]:
model.fit(X, y)
coefs = pd.Series(model[-1].coef_[0], index=FEATURES)
print(coefs.round(2).sort_values())

decay_min           0.26
dist_from_centre    0.32
log_flux            0.67
dtype: float64


In [230]:
for feats in [["log_flux", "dist_from_centre"],
              ["log_flux", "dist_from_centre", "duration_min"],
              ["log_flux", "dist_from_centre", "decay_min"]]:
    a = cross_val_score(model, flares[feats], y, cv=cv, scoring="roc_auc").mean()
    print(f"{a:.3f}  {feats}")

0.693  ['log_flux', 'dist_from_centre']
0.700  ['log_flux', 'dist_from_centre', 'duration_min']
0.708  ['log_flux', 'dist_from_centre', 'decay_min']


In [231]:
model.fit(flares[FEATURES], y)
print(pd.Series(model[-1].coef_[0], index=FEATURES).round(2).sort_values())

decay_min           0.26
dist_from_centre    0.32
log_flux            0.67
dtype: float64


In [232]:
print(flares.dtypes)
print("Rows:", len(flares))
print("Year range:", flares.event_starttime.min(), "→", flares.event_starttime.max())
print("Missing locations:", flares.hgs_x.isna().sum())
print("Label counts:")
print(flares.cme.value_counts())
print("CME catalogue years:", cmes.cme_time.dt.year.min(), "→", cmes.cme_time.dt.year.max())
print("CMEs:", len(cmes))

event_starttime     datetime64[ns]
event_peaktime      datetime64[ns]
event_endtime       datetime64[ns]
fl_goescls                  object
hgs_x                      float64
hgs_y                      float64
flare_pa                   float64
dist_from_centre           float64
cme                          int64
peak_flux                  float64
log_flux                   float64
duration_min               float64
rise_min                   float64
decay_min                  float64
dtype: object
Rows: 698
Year range: 2010-01-19 20:23:00 → 2020-11-29 12:34:00
Missing locations: 0
Label counts:
cme
0    423
1    275
Name: count, dtype: int64
CME catalogue years: 1996 → 2026
CMEs: 43140


## Summary

**Task.** Predict whether an M- or X-class solar flare launched a CME.
Warm-up for the main project (predicting IMF Bz at L1 from CME flux
rope orientation), built to develop the catalogue-matching pipeline on
a simpler, well-studied problem.

**Data.** 708 M/X flares from GOES via HEK, 2010–2020 (2018–2019 had
none; 2020 had 2). Flare locations from HEK where available, otherwise
filled from SSW Latest Events matched on peak time within 10 minutes.
This recovered 305 of 315 missing locations; the remaining 10 flares
were dropped, leaving 698. CMEs from the CDAW LASCO catalogue.

**Labels.** A flare is labelled 1 if a CME first appeared in LASCO
within 60 minutes of flare start and within 45° of the flare's position
angle, or was a halo CME. This gives a 39% association rate, consistent
with published M/X-class figures.

Label quality was estimated by shifting all flare times by ±12 and ±24
hours and rerunning the match: any hit then must be coincidental. The
false-match rate is 11% (range 9–12% across four shifts), implying
roughly 28% of flares have a genuine association and about 1 in 3.5
positive labels is spurious. The window and angle were chosen from this
real-vs-chance trade-off, not from model performance: 45 min was
marginally purer but lost a fifth of real positives, and 90 min added
mostly coincidences.

**Model.** Logistic regression with feature standardisation, evaluated
by 5-fold stratified cross-validation.

**Result.** AUC 0.71 (folds 0.68–0.76) against a 0.61 majority-class
baseline. Accuracy 0.67.

| Feature | Coefficient |
|---|---|
| log peak flux | 0.67 |
| distance from disk centre | 0.32 |
| decay time | 0.26 |

Flare brightness dominates. Position is second, consistent with limb
CMEs being easier for LASCO to detect. Decay time is a modest
contributor: brightness and position alone give AUC 0.693, so it adds
about 0.015.

Duration and rise time correlate at 0.95 — GOES end times are defined
relative to peak flux, so they are partly the same measurement. Fitted
together their coefficients were misleading (+0.79 and −0.65, acting as
a difference). Their difference, decay time, was the better single
feature: it beat duration on all five cross-validation seeds tried, by
0.004–0.008 AUC.

**Limitations.**
- Labels are noisy (~11% false-match rate), which caps achievable AUC.
- The sample is concentrated in 2011–2015; cycle 24 was weak, so
  2010 and 2016–2020 contribute little. The model mostly reflects
  solar-maximum conditions.
- Position angle matching is unreliable for disk-centre flares, whose
  CMEs often appear as halos and are accepted automatically.
- The decay-time advantage is small enough that another dataset could
  reverse it. Seed reshuffling tests split stability, not
  generalisation.
- This is a well-studied problem; the value here is the pipeline, not
  the result.

**Carries into the main project.** The `merge_asof` catalogue join, the
time-shift control for estimating false-match rates, and the practice
of choosing matching parameters on label quality rather than model
score. All three are needed to link CMEs to their ICME arrivals at L1.

## Analysing benefits of complexity


In [233]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0)
print(cross_val_score(rf, X, y, cv=cv, scoring="roc_auc").mean().round(3))

0.668


In [234]:
from sklearn.model_selection import learning_curve
sizes, train, test = learning_curve(model, X, y, cv=cv, scoring="roc_auc",
                                    train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0])
for s, te in zip(sizes, test.mean(axis=1)):
    print(f"{s:>4} samples: AUC {te:.3f}")

 111 samples: AUC 0.701
 223 samples: AUC 0.706
 334 samples: AUC 0.703
 446 samples: AUC 0.705
 558 samples: AUC 0.708
